In [94]:
# Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

## Load Cluster Node Profiles

In [95]:
def load_node_profile(node):
    filepath = f'../../tool/infrastructure-profiler/profiles/{node}.csv'

    with open(filepath) as f:
        data = f.readlines()[1].split(',')

    return {
        'ram': int(data[1].strip()),
        'cores': int(data[2].strip()),
        'io_read': float(data[3].strip()),
        'io_write': float(data[4].strip()),
        'sysbench': float(data[5].strip()),
        'z7b': float(data[6].strip()),
        'bench_time': float(data[7].strip())
    }

In [96]:
nodes = ['aws-metal-c5n', 'aws-metal-z1d', 'gpgnode-16', 'gpgnode-22', 'tu-06', 'hu-c40', 'hu-c42']
node_profiles = {}

for node in nodes:
    profile = load_node_profile(node)
    node_profiles[node] = profile

In [97]:
def combine_df(input_df, trace_df):
    rows = []
    assert len(trace_df) == len(input_df['name'].unique())

    for name, group in input_df.groupby('name'):
        no_files = str(len(group))
        filepaths = group['path']
        paths = ','.join([filepath.split('/')[-1].split('.')[-1] for filepath in filepaths])
        total_size = group['size'].sum() / 1073741824  # B to GiB
        task_record = trace_df[trace_df['name'] == name].iloc[0]
        row = [task_record['task_id'], task_record['name'], task_record['%cpu'], 
               task_record['start'], task_record['complete'], task_record['complete'] - task_record['start'], 
               no_files, paths, total_size, task_record['name'].split(':')[-1].split(' ')[0]]
        rows.append(row)

    combined = pd.DataFrame(rows, columns=['id', 'name', '%cpu', 'start', 'complete', 'runtime', 'no_files', 'files', 'input_gib', 'short_name'])
    return combined

In [98]:
def load_profiling_runs(node, workflows, samples):
    node_profile = node_profiles[node]
    runs = {}

    for workflow, samples in zip(workflows, samples):
        runs[workflow] = {}

        for sample in samples:
            sample_dir = f'{node}/{workflow}/{sample}/'
            input_df = pd.read_csv(sample_dir + 'input.csv', sep=';')
            trace_df = pd.read_csv(sample_dir + 'trace.csv')
            runs[workflow][sample] = combine_df(input_df, trace_df)

    return (node_profile, runs)

In [99]:
def load_workflow_execution(cluster, workflow, iteration):
    execution_dir = f'../{cluster}/{workflow}/{iteration}/'
    input_df = pd.read_csv(execution_dir + 'input.csv', sep=';')
    trace_df = pd.read_csv(execution_dir + 'trace.csv')
    return combine_df(input_df, trace_df)

In [100]:
def load_power_model(node, setting=''):
    if setting != '':
        filename = f'{setting}-model.txt'
    else: 
        filename = 'model.txt'

    with open(f'../../tool/infrastructure-profiler/raw/{node}/{filename}', 'r') as f:
        data = [line.strip() for line in f.readlines()[1:]]

    coeff = float(data[0].split(',')[0][2:].strip())
    inter = float(data[0].split(',')[1][:-2].strip())
    draw = float(data[1].split(':')[1].strip())

    if draw == 0:
        draw = 0.392

    return (coeff, inter, draw)


## Predict from TU-06

In [101]:
TU_06 = 'tu-06'
TU_06_workflows = ['rnaseq', 'chipseq']
TU_06_samples = [['sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(TU_06_profile, TU_06_runs) = load_profiling_runs(TU_06, TU_06_workflows, TU_06_samples)
TU_06_model = load_power_model(TU_06)

In [102]:
HU_40 = 'hu-c40'
HU_40_workflows = ['rnaseq', 'chipseq']
HU_40_samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(HU_40_profile, HU_40_runs) = load_profiling_runs(HU_40, HU_40_workflows, HU_40_samples)
HU_40_model = load_power_model(HU_40, 'schedutil')

In [103]:
HU_42 = 'hu-c42'
HU_42_workflows = ['rnaseq', 'chipseq']
HU_42_samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(HU_42_profile, HU_42_runs) = load_profiling_runs(HU_42, HU_42_workflows, HU_42_samples)
HU_42_model = load_power_model(HU_42, 'schedutil')

In [104]:
# identify real executions on hu-cluster
hu_rnaseq = load_workflow_execution('hu-cluster', 'rnaseq', 1)
hu_chipseq = load_workflow_execution('hu-cluster', 'chipseq', 1)

In [131]:
# hu-c40 -- rnaseq sample_d profiling 
comb_rnaseq_sample_d = HU_40_runs['rnaseq']['sample_d']

# q: how to use multiple runs i.e. diff samples to create this training dataset?
linear_regression = LinearRegression()
linear_regression.fit(comb_rnaseq_sample_d[['input_gib', 'no_files']], comb_rnaseq_sample_d['runtime'].values)

# use model to predict duration based on input size from a FULL WORKFLOW RUN ON
# - the same cluster node type
# - a different cluster node type

for _, record in hu_rnaseq.iterrows():
    print(record['input_gib'], record['runtime'], record['no_files'], record['files'])
    break

# Example of how to use the model to predict duration for a new input size
new_input_size = 1.580382063984871  # value in Bytes
predicted_duration = linear_regression.predict([[new_input_size, 2]])
print('Linear Regression')
print(f'input size {new_input_size}GiB | duration {predicted_duration[0]:.0f} ms')


1.580382063984871 14000 2 bai,bam
Linear Regression
input size 1.580382063984871GiB | duration 62989 ms


/usr/lib/python3/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [106]:
def get_task_data_from_experiment(task_name, experiment):
    task_data = experiment[experiment['short_name'] == task_name]
    key_data = task_data[['input_gib', 'runtime']]
    key_data.columns = ['x', 'y']
    key_data.reset_index().drop('index', axis=1)
    return key_data

In [132]:
# train models for all tasks in rnaseq from profiling runs
tasks = HU_40_runs['rnaseq']['sample_a']['short_name'].unique()
task_models = {}

for task in tasks:
    task_data = pd.DataFrame(columns=['x', 'y'], dtype='float64')
    for sample in ['sample_a', 'sample_b', 'sample_c', 'sample_d']:
        experiment_data = get_task_data_from_experiment(task, HU_40_runs['rnaseq'][sample])
        task_data = pd.concat([task_data, experiment_data])

    lr = LinearRegression()
    task_models[task] = lr.fit(task_data['x'].values.reshape(-1, 1), task_data['y'].values) 

# make prediction using trained models
overall_error = []
missing_tasks = []

for _, record in hu_rnaseq.iterrows():
    task_name = record['short_name']
    data_size = record['input_gib']
    actual_runtime = record['runtime']

    if task_name in task_models.keys():
        predicted_runtime = task_models[task_name].predict([[data_size]])[0]

        if actual_runtime > 0:
            if predicted_runtime > 0:
                error = (abs(predicted_runtime - actual_runtime) / actual_runtime) * 100 
                overall_error.append(error)
            #else:
                #print(task_name, data_size, predicted_runtime, actual_runtime, error)  # to be investigated
            #print(f'{task_name} with data {data_size}B --> pred {predicted_runtime}ms actu {actual_runtime}ms erro {error}%')
    else:
        missing_tasks.append(task_name)

print('avg. error', np.mean(overall_error))

avg. error 39.035081950780324


In [128]:
class MedianModel:
    def __init__(self, median):
        self.median = median
    
    def predict(self, x):
        x = np.asarray(x)
        value = x.shape[0]
        return np.full(value, self.median)

In [ ]:
# train models for all tasks in rnaseq from profiling runs on TU-06
tasks = TU_06_runs['rnaseq']['sample_b']['short_name'].unique()
task_models = {}

for task in tasks:
    task_data = pd.DataFrame(columns=['x', 'y'], dtype='float64')
    for sample in ['sample_b', 'sample_c', 'sample_d']:
        experiment_data = get_task_data_from_experiment(task, TU_06_runs['rnaseq'][sample])
        task_data = pd.concat([task_data, experiment_data])

    # only works when we actively vary the input size i.e. 4x similar sizes is useless (!) --
    # @Vasilis I'm going to do additional profiling runs with smaller samples / bigger samples to get this data so we can use these approaches :) 
    #pearson = np.corrcoef(task_data['x'].to_numpy(), task_data['y'].to_numpy())[0, 1]

    #if np.isnan(pearson) or pearson < 0.75:
    #    task_models[task] = MedianModel(np.median(task_data['y'].to_numpy()))
    #else:
    lr = LinearRegression()
    task_models[task] = lr.fit(task_data['x'].values.reshape(-1, 1), task_data['y'].values) 
        # alt -- Lotaru used a BayesianRidge
        #     model = BayesianRidge(fit_intercept=True)
        #     model.fit(x.reshape(-1, 1), y)

# translation factor from TU to HU 
target = HU_40_profile
local = TU_06_profile
print('local', local)
print('target', target)
factor = (0.25 * local['z7b'] / target['z7b']) + (0.25 * local['sysbench'] / target['sysbench']) + (0.25 * local['io_write'] / target['io_write']) + (0.25 * local['io_read'] / target['io_read'])
print(factor)

# make prediction using trained models (TU-06 to HU-nodes)
overall_error = []
missing_tasks = []
hu_rnaseq = load_workflow_execution('hu-cluster', 'rnaseq', 1)

for _, record in hu_rnaseq.iterrows():
    task_name = record['short_name']
    data_size = record['input_gib']
    actual_runtime = record['runtime']

    if task_name in task_models.keys():
        predicted_runtime = task_models[task_name].predict([[data_size]])[0] * factor

        if actual_runtime > 0:
            if predicted_runtime > 0:
                error = (abs(predicted_runtime - actual_runtime) / actual_runtime) * 100 
                if error <= 100:
                    overall_error.append(error)
            #else:
                #print(task_name, data_size, predicted_runtime, actual_runtime, error)  # to be investigated
            print(f'{task_name}: {data_size}GiB -> Predicted {predicted_runtime / 1000:.2f}s Actual {actual_runtime / 1000:.2f}s (Error {error:.1f}%)')
    else:
        missing_tasks.append(task_name)

print('avg. error', np.mean(overall_error))

local {'ram': 128, 'cores': 32, 'io_read': 3500.0, 'io_write': 4717.0, 'sysbench': 601.07, 'z7b': 3014.0, 'bench_time': 0.07366666666666667}
target {'ram': 256, 'cores': 32, 'io_read': 1362.0, 'io_write': 952.0, 'sysbench': 1056.95, 'z7b': 2711.0, 'bench_time': 23.160666666666668}
2.3012581629028697
SAMTOOLS_FLAGSTAT: 1.580382063984871GiB -> Predicted 15.71s Actual 14.00s (Error 12.2%)
SAMTOOLS_FLAGSTAT: 1.5803867308422923GiB -> Predicted 15.71s Actual 14.00s (Error 12.2%)
SAMTOOLS_FLAGSTAT: 1.5803545638918877GiB -> Predicted 15.71s Actual 16.00s (Error 1.8%)
SAMTOOLS_FLAGSTAT: 1.646299010142684GiB -> Predicted 16.21s Actual 14.00s (Error 15.8%)
SAMTOOLS_FLAGSTAT: 1.1218057507649064GiB -> Predicted 12.28s Actual 10.00s (Error 22.8%)
SAMTOOLS_FLAGSTAT: 1.3909376477822661GiB -> Predicted 14.29s Actual 13.00s (Error 10.0%)
SAMTOOLS_FLAGSTAT: 1.8837378099560738GiB -> Predicted 17.99s Actual 17.00s (Error 5.8%)
SAMTOOLS_FLAGSTAT: 1.7031185328960419GiB -> Predicted 16.63s Actual 14.00s (Erro

In [109]:
# THE PLAN

# also - look at tasks that don't scale linearly (pearsson)
# investigate tasks with prediction of negative runtime?
# investigate tasks with runtime of 0

# prediction error for runtime should be ~15%, not up at 40/50%

# confirm on whether comparing dynamic power consumption for individual tasks, and then overall should consider static over periods of active execution (using traces)

# identify the governor and compare energy estimation for each workflow using ichnos (!)
# have these task-level estimates of power consumed, and total inc. and exc. static power consumption for the nodes
# automate the process

# next step, combine the predicted runtime with estimation of energy using node power models
# start to compare task-level estimates with traces


In [ ]:
# Make a simple svm, linear regression and random forest regression model to predict y from x

# try each in the comparison

# models = {
#     'SVR': SVR(),
#     'RandomForest': RandomForestRegressor(),
#     'LinearRegression': LinearRegression(),
#     'BayesianRidge': BayesianRidge()
# }

# mse = mean_squared_error(y_test, y_pred) 
# print(f'Linear Regression MSE: {mse}')
